# 03 - Train the acoustic model

About 7 to 9 hours on a 4090, roughly 5 to 7 dollars on RunPod.

Every `sample_every` steps the probe sentences are rendered to TensorBoard, so
you can *listen* to whether homographs are pronounced correctly instead of
guessing from a loss curve.

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.environ["PYTHONIOENCODING"] = "utf-8"

# ---------------------------------------------------------------------------
# PICK YOUR EXPERIMENT HERE. This is the only line to change.
#
#   configs/exp0_small.yaml     2013 clips, ~2 GB   -> proves the pipeline,
#                                                     runs on a 6 GB GPU
#   configs/exp1_egyptian.yaml  15.6k clips, 68 h   -> the real run
# ---------------------------------------------------------------------------
CONFIG = "configs/exp0_small.yaml"

from adaptts.utils.config import load_config
from adaptts.utils.logging_utils import setup_logging
setup_logging()
cfg = load_config(CONFIG)
print("repo   :", REPO)
print("config :", CONFIG, "->", cfg.name)
print("dataset:", cfg.paths.hf_dataset_id)

# The homographs named in the brief, read from the probe file so the notebooks
# never hardcode a word list of their own.
import json as _json
PROBE_WORDS = sorted({
    w for _s in _json.load(open("assets/probe_sentences.json", encoding="utf-8"))["sentences"]
    for w in _s["focus"].split(" / ") if w and w != "none"
})
print("probe  :", " ".join(PROBE_WORDS))

## Check the model size before committing GPU hours

In [ ]:
from adaptts.models.acoustic import AcousticModel
from adaptts.text.vocab import CharVocab


vocab = CharVocab.load(cfg.paths.charvocab_path)
m = AcousticModel(
    len(vocab), n_quantizers=cfg.audio.n_quantizers, codebook_size=cfg.audio.codebook_size,
    d_model=cfg.acoustic.d_model, n_layers=cfg.acoustic.n_layers, n_heads=cfg.acoustic.n_heads,
    d_ff=cfg.acoustic.d_ff, text_d_model=cfg.acoustic.text_d_model,
    text_n_layers=cfg.acoustic.text_n_layers, text_n_heads=cfg.acoustic.text_n_heads,
    depth_d_model=cfg.acoustic.depth_d_model, depth_n_layers=cfg.acoustic.depth_n_layers,
    depth_n_heads=cfg.acoustic.depth_n_heads, speaker_dim=cfg.acoustic.speaker_dim,
    max_codes=cfg.discovery.max_codes_per_word, pc_embed_dim=cfg.acoustic.pc_embed_dim,
    exit_layers=cfg.acoustic.exit_layers, pad_id=vocab.pad_id,
)
n = sum(p.numel() for p in m.parameters())
print(f"acoustic model: {n / 1e6:.1f}M parameters")
print(f"fp32 weights:   {n * 4 / 1024 ** 2:.0f} MB")
print(f"total deployed with the frozen Mimi decoder: about {(n + 25e6) / 1e6:.0f}M")
del m

## TensorBoard

Watch `train/acc_q0` for the coarse RVQ level, and the `probe/` audio tab.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $cfg.paths.tb_dir --port 6006 --bind_all

## Train

If the pod restarts, resume with
`--resume runs/exp1/checkpoints/acoustic/last.pt`.

In [ ]:
!python scripts/train_acoustic.py --config $CONFIG

## Quick machinery check

Before listening critically, confirm the parts are working: real samples, a
measurable difference between exit depths, and an override that actually
changes the output. Audio quality at this point depends entirely on how long
the model trained.

In [ ]:
!python scripts/smoke_generate.py --config $CONFIG --device cpu

## Listen to the probe set

In [ ]:
import json
from IPython.display import Audio, display
from adaptts.infer.pipeline import AdapTTS

tts = AdapTTS.from_checkpoints(CONFIG, device="cpu")
probes = json.load(open("assets/probe_sentences.json", encoding="utf-8"))["sentences"]

for p in probes:
    plan = tts.analyze(p["text"])
    wav, st = tts.synthesize(plan)
    print()
    print(f"=== {p['tag']} | expected {p['expected']} ===")
    print("   ", p["text"])
    print(f"    depth {st['depth']}  difficulty {st['sentence_difficulty']:.2f}  "
          f"RTF {st['real_time_factor']:.2f}")
    display(Audio(wav, rate=st["sample_rate"]))

## The controllability demo

The same sentence rendered with each reading, by overriding the code. No
diacritics are typed anywhere.

In [ ]:
text = "انا شوفت علم مصر بيرفرف"
plan = tts.analyze(text)
print(plan)

w = [x for x in plan.hard_words if x.word == "علم"][0]
for c in range(w.n_codes):
    plan.set_code("علم", c)
    wav, st = tts.synthesize(plan)
    print()
    print(f"--- علم forced to code {c} ---")
    display(Audio(wav, rate=st["sample_rate"]))

## Adaptive depth: measure the saving

An easy sentence should route to a shallow exit and be measurably faster.

In [ ]:
import time

easy = "الجو النهارده حلو جدا و الشمس طالعة"
hard = "انا كنت مصر على ان مصر عندها امكانيات تخليها تتفوق على دول"

for name, txt in [("easy", easy), ("hard", hard)]:
    plan = tts.analyze(txt)
    t0 = time.perf_counter()
    wav, st = tts.synthesize(plan)
    dt = time.perf_counter() - t0
    print(f"{name:5s} difficulty {plan.sentence_difficulty:.3f} -> depth {st['depth']:2d}  "
          f"{dt:.2f}s for {st['audio_seconds']:.1f}s audio  RTF {st['real_time_factor']:.2f}")

In [ ]:
# Force each depth on the same sentence to isolate the compute saving.
plan = tts.analyze(hard)
for d in cfg.acoustic.exit_layers:
    plan.set_depth(d)
    t0 = time.perf_counter()
    wav, st = tts.synthesize(plan)
    dt = time.perf_counter() - t0
    print(f"depth {d:2d}: {dt:.2f}s  RTF {st['real_time_factor']:.2f}")
    display(Audio(wav, rate=st["sample_rate"]))